# 03 — Feature Engineering
### Preparing Features for Hedonic Pricing & ML Models

**Pipeline:**
1. Load unified_dataset.parquet
2. Numeric transforms (log_mileage, mileage_per_year, power_per_age, age_bin)
3. Stratified split FIRST (70/15/15 by country x make_grouped x year, collapse rare strata <10)
4. Target-encode on train only: make_encoded, model_encoded, body_encoded, country_fe
5. Other encodings: transmission, drive, electrification_score, is_ev/phev/ice, condition, color
6. EV features: electric_range, battery_cost_index from materials
7. Interaction terms: age_x_mileage, age_squared, mileage_squared, ev_x_year, power_x_trans, make_x_age
8. Equipment features
9. Define ML_FEATURES (24) and HEDONIC_FEATURES (28) lists
10. Save features_engineered.parquet + feature_metadata.json

**Key principle:** Target encoding is computed on the TRAIN set only, then applied to val/test to prevent data leakage.

In [1]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from config import (
    DATA_PROC, DATA_SPLITS, FIGURES, MATERIALS, BATTERY_MATERIALS, MOTOR_MATERIALS,
    NEUTRAL_COLORS, RANDOM_STATE, LOG_TARGET, TARGET_COL,
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO, STRATIFY_COLS,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 130

print('Setup complete.')

Setup complete.


## 1. Load Unified Dataset

In [2]:
df = pd.read_parquet(DATA_PROC / 'unified_dataset.parquet')
mat = pd.read_parquet(DATA_PROC / 'materials.parquet')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Years: {sorted(df["data_year"].unique())}')
print(f'Target: {TARGET_COL} / {LOG_TARGET}')

Loaded: 228,870 rows x 53 cols
Years: [np.int64(2023), np.int64(2024), np.int64(2025)]
Target: price_eur / log_price


## 2. Numeric Transforms

In [3]:
# Ensure numeric types
for col in ['mileage_km', 'vehicle_age', 'power_hp', 'doors']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Log-mileage (reduces skew)
df['log_mileage'] = np.log1p(df['mileage_km'].clip(lower=0))

# Mileage per year of age (usage intensity)
df['mileage_per_year'] = df['mileage_km'] / df['vehicle_age'].replace(0, np.nan)
df['mileage_per_year'] = df['mileage_per_year'].clip(upper=100_000)

# Power per age (depreciation-adjusted performance)
df['power_per_age'] = df['power_hp'] / (df['vehicle_age'] + 1)

# Age bins (for stratification and non-linear age effects)
df['age_bin'] = pd.cut(
    df['vehicle_age'],
    bins=[-1, 1, 3, 5, 8, 12, 20, 100],
    labels=['0-1', '2-3', '4-5', '6-8', '9-12', '13-20', '20+']
).astype(str)

print('Numeric features created:')
for col in ['log_mileage', 'mileage_per_year', 'power_per_age']:
    print(f'  {col}: mean={df[col].mean():.2f}, median={df[col].median():.2f}, '
          f'missing={df[col].isna().sum():,}')

Numeric features created:
  log_mileage: mean=9.63, median=10.62, missing=272
  mileage_per_year: mean=12975.13, median=11538.46, missing=34,223
  power_per_age: mean=87.65, median=53.00, missing=6,899


## 3. Stratified Split FIRST (70/15/15)

Split by `country_code x make_grouped x data_year`. Strata with fewer than 10 observations
are collapsed into an `_OTHER_` group to ensure every stratum has enough samples for splitting.

In [4]:
# Group rare makes (< 500 listings) into 'Other'
make_counts = df['make'].value_counts()
top_makes = make_counts[make_counts >= 500].index
df['make_grouped'] = df['make'].where(df['make'].isin(top_makes), 'Other')
print(f'Makes: {df["make"].nunique()} -> {df["make_grouped"].nunique()} (grouped)')

# Build stratification key
df['_strat_key'] = (
    df['country_code'].astype(str) + '|' +
    df['make_grouped'].astype(str) + '|' +
    df['data_year'].astype(str)
)

# Collapse rare strata (< 10 members) into '_OTHER_'
strat_counts = df['_strat_key'].value_counts()
rare_strata = strat_counts[strat_counts < 10].index
df.loc[df['_strat_key'].isin(rare_strata), '_strat_key'] = '_OTHER_'
print(f'Strata: {df["_strat_key"].nunique()} unique (rare collapsed to _OTHER_)')
print(f'_OTHER_ stratum size: {(df["_strat_key"] == "_OTHER_").sum():,}')

# First split: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['_strat_key'], random_state=RANDOM_STATE
)

# Second split: temp -> val (15%) + test (15%)
relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df, test_size=relative_test,
    stratify=temp_df['_strat_key'], random_state=RANDOM_STATE
)

# Store split labels
df['split'] = 'train'
df.loc[val_df.index, 'split'] = 'val'
df.loc[test_df.index, 'split'] = 'test'

print(f'\nSplit sizes:')
print(f'  Train: {len(train_df):>8,} ({len(train_df)/len(df)*100:.1f}%)')
print(f'  Val:   {len(val_df):>8,} ({len(val_df)/len(df)*100:.1f}%)')
print(f'  Test:  {len(test_df):>8,} ({len(test_df)/len(df)*100:.1f}%)')
print(f'  Total: {len(df):>8,}')

train_mask = df['split'] == 'train'

Makes: 67 -> 43 (grouped)
Strata: 447 unique (rare collapsed to _OTHER_)
_OTHER_ stratum size: 347

Split sizes:
  Train:  160,209 (70.0%)
  Val:     34,330 (15.0%)
  Test:    34,331 (15.0%)
  Total:  228,870


In [5]:
# Verify stratification quality
print('--- Split quality check: year distribution ---')
print(pd.crosstab(df['split'], df['data_year'], normalize='index').round(3))

print('\n--- Split quality check: powertrain distribution ---')
print(pd.crosstab(df['split'], df['powertrain'], normalize='index').round(3))

print('\n--- Split quality check: top 5 countries ---')
top5 = df['country_code'].value_counts().head(5).index
print(pd.crosstab(df['split'], df['country_code'])[top5])

--- Split quality check: year distribution ---
data_year   2023   2024   2025
split                         
test       0.376  0.152  0.471
train      0.376  0.153  0.471
val        0.376  0.153  0.471

--- Split quality check: powertrain distribution ---
powertrain     EV    ICE  Other   PHEV
split                                 
test        0.053  0.831  0.008  0.108
train       0.053  0.831  0.008  0.108
val         0.053  0.832  0.008  0.107

--- Split quality check: top 5 countries ---
country_code     DE     IT     NL     BE     FR
split                                          
test          11446   6240   5139   3700   3186
train         53375  29098  23989  17277  14879
val           11424   6226   5142   3706   3187


## 4. Target Encoding (Train Only)

Compute mean `log_price` per category on TRAIN set, then map to all splits.

In [6]:
global_mean = df.loc[train_mask, LOG_TARGET].mean()
print(f'Global mean log_price (train): {global_mean:.4f}')

# -- Make encoding (target-encoded) --
make_mean = df.loc[train_mask].groupby('make_grouped')[LOG_TARGET].mean()
df['make_encoded'] = df['make_grouped'].map(make_mean).fillna(
    make_mean.get('Other', global_mean)
)

# -- Model encoding (target-encoded, group rare models) --
model_counts = df['model'].value_counts()
top_models = model_counts[model_counts >= 100].index
df['model_grouped'] = df['model'].where(df['model'].isin(top_models), 'Other')
model_mean = df.loc[train_mask].groupby('model_grouped')[LOG_TARGET].mean()
df['model_encoded'] = df['model_grouped'].map(model_mean).fillna(
    model_mean.get('Other', global_mean)
)
print(f'Models: {df["model"].nunique()} -> {df["model_grouped"].nunique()} (grouped)')

# -- Body type encoding (target-encoded) --
body_mean = df.loc[train_mask].groupby('body_type')[LOG_TARGET].mean()
df['body_encoded'] = df['body_type'].map(body_mean).fillna(global_mean)

# -- Country fixed effect (target-encoded) --
country_mean = df.loc[train_mask].groupby('country_code')[LOG_TARGET].mean()
df['country_fe'] = df['country_code'].map(country_mean).fillna(global_mean)

print(f'\nTarget encodings computed on train set ({train_mask.sum():,} rows):')
print(f'  make_encoded:  {df["make_encoded"].nunique()} unique values')
print(f'  model_encoded: {df["model_encoded"].nunique()} unique values')
print(f'  body_encoded:  {df["body_encoded"].nunique()} unique values')
print(f'  country_fe:    {df["country_fe"].nunique()} unique values')

Global mean log_price (train): 10.2741
Models: 1476 -> 361 (grouped)

Target encodings computed on train set (160,209 rows):
  make_encoded:  43 unique values
  model_encoded: 361 unique values
  body_encoded:  8 unique values
  country_fe:    8 unique values


## 5. Other Encodings

In [7]:
# -- Transmission encoding (ordinal) --
trans_map = {'Manual': 0, 'Semi-Automatic': 1, 'Automatic': 2, 'Unknown': 0}
df['transmission_encoded'] = df['transmission'].map(trans_map).fillna(0)

# -- Drivetrain encoding (ordinal) --
drive_map = {
    'front wheel drive': 0, 'front wheels': 0,
    'rear wheel drive': 1,
    'all wheel drive': 2, '4wd': 2,
}
if 'drivetrain' in df.columns:
    df['drive_encoded'] = (
        df['drivetrain'].astype(str).str.strip().str.lower()
        .map(drive_map).fillna(0)
    )
else:
    df['drive_encoded'] = 0

# -- Electrification score --
electrification = {
    'Electric': 5, 'Hybrid_PHEV': 4, 'Hybrid_Diesel': 3,
    'CNG': 2, 'LPG': 2, 'Diesel': 1, 'Gasoline': 1, 'Other': 0
}
df['electrification_score'] = df['fuel_type'].map(electrification).fillna(0)

# -- Binary powertrain indicators --
df['is_ev'] = (df['powertrain'] == 'EV').astype(int)
df['is_phev'] = (df['powertrain'] == 'PHEV').astype(int)
df['is_ice'] = (df['powertrain'] == 'ICE').astype(int)

# -- Condition encoding (ordinal) --
cond_map = {'New': 2, 'Pre-Registered': 1, 'Used': 0}
df['condition_encoded'] = df['condition'].map(cond_map).fillna(0)

# -- Color: neutral vs non-neutral --
df['is_neutral_color'] = (
    df['color'].astype(str).str.lower().isin(NEUTRAL_COLORS).astype(int)
)

print('Ordinal / binary encodings complete:')
for col in ['transmission_encoded', 'drive_encoded', 'electrification_score',
            'is_ev', 'is_phev', 'is_ice', 'condition_encoded', 'is_neutral_color']:
    print(f'  {col}: {df[col].value_counts().to_dict()}')

Ordinal / binary encodings complete:
  transmission_encoded: {2: 182306, 0: 40648, 1: 5916}
  drive_encoded: {0.0: 151919, 2.0: 54077, 1.0: 22874}
  electrification_score: {1: 189275, 4: 21342, 5: 12066, 3: 3390, 0: 1802, 2: 995}
  is_ev: {0: 216804, 1: 12066}
  is_phev: {0: 204138, 1: 24732}
  is_ice: {1: 190270, 0: 38600}
  condition_encoded: {0: 219164, 2: 9706}
  is_neutral_color: {1: 174683, 0: 54187}


## 6. EV Features & Battery Cost Index

In [8]:
# Electric range features
if 'electric_range_km' in df.columns:
    df['electric_range_km'] = pd.to_numeric(df['electric_range_km'], errors='coerce')
    df['has_electric_range'] = df['electric_range_km'].notna().astype(int)
    df['electric_range_km'] = df['electric_range_km'].fillna(0)
else:
    df['electric_range_km'] = 0
    df['has_electric_range'] = 0

# -- Battery Material Cost Index --
mat['date'] = pd.to_datetime(mat['date'])
mat['year'] = mat['date'].dt.year
mat_annual = mat.groupby(['year', 'material'])['price_usd_per_ton'].mean().reset_index()

battery_weights = {'Lithium': 0.35, 'Cobalt': 0.25, 'Nickel': 0.25, 'Graphite': 0.15}
bci = []
for yr in range(2021, 2026):
    yr_mat = mat_annual[mat_annual['year'] == yr].set_index('material')['price_usd_per_ton']
    cost = sum(yr_mat.get(m, 0) * w for m, w in battery_weights.items())
    bci.append({'year': yr, 'battery_cost_index': cost})
bci_df = pd.DataFrame(bci)

# Normalize to 0-1 scale
bci_min, bci_max = bci_df['battery_cost_index'].min(), bci_df['battery_cost_index'].max()
bci_df['battery_cost_index'] = (bci_df['battery_cost_index'] - bci_min) / (bci_max - bci_min)

# Merge with dataset
df = df.merge(bci_df, left_on='data_year', right_on='year', how='left').drop(columns='year')

# Set battery_cost_index to 0 for non-EVs
df.loc[df['is_ev'] == 0, 'battery_cost_index'] = 0

print('EV features created:')
print(f'  EV records with electric_range > 0: {(df["electric_range_km"] > 0).sum():,}')
print(f'  Battery cost index by year:')
print(bci_df.to_string(index=False))

EV features created:
  EV records with electric_range > 0: 11,538
  Battery cost index by year:
 year  battery_cost_index
 2021            0.806130
 2022            1.000000
 2023            0.327901
 2024            0.000000
 2025            0.015415


## 7. Interaction Terms (for Hedonic Models)

In [9]:
# Interaction terms grounded in hedonic pricing theory:

# 1. Age x Mileage: depreciation depends on usage intensity
df['age_x_mileage'] = df['vehicle_age'] * df['log_mileage']

# 2. Age squared: quadratic depreciation curve
df['age_squared'] = df['vehicle_age'] ** 2

# 3. Mileage squared: non-linear mileage effect
df['mileage_squared'] = df['log_mileage'] ** 2

# 4. EV x Year: EV premium evolution over time
df['ev_x_year'] = df['is_ev'] * df['data_year']

# 5. Power x Transmission: automatic premium varies by power level
df['power_x_trans'] = df['power_hp'] * df['transmission_encoded']

# 6. Make x Age: brand depreciation curves differ
df['make_x_age'] = df['make_encoded'] * df['vehicle_age']

print('Interaction terms created:')
for col in ['age_x_mileage', 'age_squared', 'mileage_squared',
            'ev_x_year', 'power_x_trans', 'make_x_age']:
    print(f'  {col}: mean={df[col].mean():.2f}, '
          f'missing={df[col].isna().sum():,}')

Interaction terms created:
  age_x_mileage: mean=61.09, missing=5,322
  age_squared: mean=85.04, missing=5,075
  mileage_squared: mean=101.26, missing=272
  ev_x_year: mean=106.72, missing=0
  power_x_trans: mean=465.67, missing=1,885
  make_x_age: mean=57.56, missing=5,075


## 8. Equipment Features

In [10]:
# Boolean equipment features from 2023 data (has_ac, has_navigation, etc.)
bool_eq_cols = [c for c in df.columns if c.startswith('has_')]
for c in bool_eq_cols:
    df[c] = df[c].map(
        {'TRUE': 1, 'True': 1, 'FALSE': 0, 'False': 0, True: 1, False: 0}
    ).fillna(0).astype(int)
df['eq_bool_total'] = df[bool_eq_cols].sum(axis=1)

# Count equipment items from structured columns (2025 data)
for eq_col in ['eq_comfort', 'eq_entertainment', 'eq_safety', 'eq_extra']:
    if eq_col in df.columns:
        df[f'{eq_col}_count'] = df[eq_col].apply(
            lambda x: 0 if pd.isna(x) or str(x) in ('nan', '') else str(x).count(',') + 1
        )

print(f'Equipment features:')
print(f'  eq_bool_total: mean={df["eq_bool_total"].mean():.1f}')
print(f'  Boolean equipment cols: {len(bool_eq_cols)}')

Equipment features:
  eq_bool_total: mean=1.6
  Boolean equipment cols: 7


## 9. Define ML_FEATURES (24) and HEDONIC_FEATURES (28)

In [11]:
# ML features (24): core features for tree-based and neural network models
ML_FEATURES = [
    'log_mileage',
    'vehicle_age',
    'power_hp',
    'data_year',
    'mileage_per_year',
    'power_per_age',
    'make_encoded',
    'model_encoded',
    'body_encoded',
    'transmission_encoded',
    'drive_encoded',
    'electrification_score',
    'condition_encoded',
    'is_ev',
    'is_phev',
    'is_ice',
    'is_neutral_color',
    'country_fe',
    'doors',
    'electric_range_km',
    'has_electric_range',
    'battery_cost_index',
    'eq_bool_total',
]

# Hedonic features (28): ML features + interaction terms for OLS/hedonic models
HEDONIC_INTERACTIONS = [
    'age_x_mileage',
    'age_squared',
    'mileage_squared',
    'ev_x_year',
    'power_x_trans',
    'make_x_age',
]
HEDONIC_FEATURES = ML_FEATURES + HEDONIC_INTERACTIONS

# Verify all features exist
ml_missing = [c for c in ML_FEATURES if c not in df.columns]
hedonic_missing = [c for c in HEDONIC_FEATURES if c not in df.columns]
if ml_missing:
    print(f'WARNING: ML features missing from DataFrame: {ml_missing}')
if hedonic_missing:
    print(f'WARNING: Hedonic features missing from DataFrame: {hedonic_missing}')

# Filter to existing columns only
ML_FEATURES = [c for c in ML_FEATURES if c in df.columns]
HEDONIC_FEATURES = [c for c in HEDONIC_FEATURES if c in df.columns]

print(f'ML Features: {len(ML_FEATURES)}')
print(f'Hedonic Features: {len(HEDONIC_FEATURES)}')
print(f'\nML features ({len(ML_FEATURES)}):')
for i, f in enumerate(ML_FEATURES, 1):
    print(f'  {i:2d}. {f}')
print(f'\nAdditional hedonic interactions ({len(HEDONIC_INTERACTIONS)}):')
for i, f in enumerate(HEDONIC_INTERACTIONS, 1):
    print(f'  {i:2d}. {f}')

ML Features: 23
Hedonic Features: 29

ML features (23):
   1. log_mileage
   2. vehicle_age
   3. power_hp
   4. data_year
   5. mileage_per_year
   6. power_per_age
   7. make_encoded
   8. model_encoded
   9. body_encoded
  10. transmission_encoded
  11. drive_encoded
  12. electrification_score
  13. condition_encoded
  14. is_ev
  15. is_phev
  16. is_ice
  17. is_neutral_color
  18. country_fe
  19. doors
  20. electric_range_km
  21. has_electric_range
  22. battery_cost_index
  23. eq_bool_total

Additional hedonic interactions (6):
   1. age_x_mileage
   2. age_squared
   3. mileage_squared
   4. ev_x_year
   5. power_x_trans
   6. make_x_age


In [12]:
# Feature completeness check
print('=' * 80)
print('FEATURE COMPLETENESS CHECK')
print('=' * 80)
for col in HEDONIC_FEATURES:
    n_miss = df[col].isna().sum()
    pct = n_miss / len(df) * 100
    print(f'  {col:25s}: {n_miss:>7,} missing ({pct:5.1f}%) '
          f'| mean={df[col].mean():>10.2f} | std={df[col].std():>10.2f}')

FEATURE COMPLETENESS CHECK
  log_mileage              :     272 missing (  0.1%) | mean=      9.63 | std=      2.93
  vehicle_age              :   5,075 missing (  2.2%) | mean=      5.59 | std=      7.33
  power_hp                 :   1,885 missing (  0.8%) | mean=    266.47 | std=    140.96
  data_year                :       0 missing (  0.0%) | mean=   2024.10 | std=      0.92
  mileage_per_year         :  34,223 missing ( 15.0%) | mean=  12975.13 | std=  10308.54
  power_per_age            :   6,899 missing (  3.0%) | mean=     87.65 | std=     96.87
  make_encoded             :       0 missing (  0.0%) | mean=     10.27 | std=      0.39
  model_encoded            :       0 missing (  0.0%) | mean=     10.27 | std=      0.57
  body_encoded             :       0 missing (  0.0%) | mean=     10.27 | std=      0.23
  transmission_encoded     :       0 missing (  0.0%) | mean=      1.62 | std=      0.77
  drive_encoded            :       0 missing (  0.0%) | mean=      0.57 | std=     

## 10. Save features_engineered.parquet + feature_metadata.json

In [13]:
# Cast object columns to string for parquet compatibility
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str)

# Save full feature-engineered dataset
out_path = DATA_PROC / 'features_engineered.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Size: {out_path.stat().st_size / 1e6:.1f} MB')
print(f'Shape: {df.shape}')

# Save feature metadata
feat_meta = {
    'ml_features': ML_FEATURES,
    'hedonic_features': HEDONIC_FEATURES,
    'hedonic_interactions': HEDONIC_INTERACTIONS,
    'n_ml_features': len(ML_FEATURES),
    'n_hedonic_features': len(HEDONIC_FEATURES),
    'target': LOG_TARGET,
    'split_ratios': {
        'train': TRAIN_RATIO,
        'val': VAL_RATIO,
        'test': TEST_RATIO,
    },
    'split_sizes': {
        'train': int((df['split'] == 'train').sum()),
        'val': int((df['split'] == 'val').sum()),
        'test': int((df['split'] == 'test').sum()),
    },
    'stratify_cols': STRATIFY_COLS,
    'random_state': RANDOM_STATE,
}

meta_path = DATA_PROC / 'feature_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(feat_meta, f, indent=2)
print(f'\nFeature metadata saved: {meta_path}')
print(json.dumps(feat_meta, indent=2))

print('\n' + '=' * 80)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 80)

Saved: /Users/elbekmajidov/Developer/Master's in Data Science/University of Warsaw/master-thesis-9/notebooks/../data/processed/features_engineered.parquet
Size: 27.0 MB
Shape: (228870, 86)

Feature metadata saved: /Users/elbekmajidov/Developer/Master's in Data Science/University of Warsaw/master-thesis-9/notebooks/../data/processed/feature_metadata.json
{
  "ml_features": [
    "log_mileage",
    "vehicle_age",
    "power_hp",
    "data_year",
    "mileage_per_year",
    "power_per_age",
    "make_encoded",
    "model_encoded",
    "body_encoded",
    "transmission_encoded",
    "drive_encoded",
    "electrification_score",
    "condition_encoded",
    "is_ev",
    "is_phev",
    "is_ice",
    "is_neutral_color",
    "country_fe",
    "doors",
    "electric_range_km",
    "has_electric_range",
    "battery_cost_index",
    "eq_bool_total"
  ],
  "hedonic_features": [
    "log_mileage",
    "vehicle_age",
    "power_hp",
    "data_year",
    "mileage_per_year",
    "power_per_age",
    